# 23 Land-Cover Ensemble

Ensemble from baseline + random forest + boosting notebooks.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import importlib

# Resolve project root in local or Kaggle runtime.
if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working")
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if (ROOT / "src").exists():
    SRC = ROOT / "src"
else:
    SRC = Path.cwd() / "src"
if str(SRC) not in sys.path and SRC.exists():
    sys.path.append(str(SRC))

if importlib.util.find_spec("laspy") is None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "laspy[lazrs]"])

from lidar_roraima.config import ProjectConfig
cfg = ProjectConfig.from_root(ROOT)
cfg.ensure_dirs()

# Kaggle dataset fallback path.
RAW_DATA_DIR = cfg.raw_data_dir
for candidate in [
    Path("/kaggle/input/lidar-roraima-parime-research/lidar_data"),
    Path("/kaggle/input/lidar-roraima-parime-research"),
]:
    if candidate.exists():
        RAW_DATA_DIR = candidate
        break

print("ROOT:", ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
TRAIN_MAX_ROWS = None  # Set integer for constrained runtime, e.g. 1200000
cfg


ROOT: C:\Users\alexy\Documents\ChatGPT_projects\LIDAR RORAIMA
RAW_DATA_DIR: C:\Users\alexy\Documents\ChatGPT_projects\LIDAR RORAIMA\lidar_data


ProjectConfig(root_dir=WindowsPath('C:/Users/alexy/Documents/ChatGPT_projects/LIDAR RORAIMA'), raw_data_dir=WindowsPath('C:/Users/alexy/Documents/ChatGPT_projects/LIDAR RORAIMA/lidar_data'), artifacts_dir=WindowsPath('C:/Users/alexy/Documents/ChatGPT_projects/LIDAR RORAIMA/artifacts'), manifests_dir=WindowsPath('C:/Users/alexy/Documents/ChatGPT_projects/LIDAR RORAIMA/artifacts/manifests'), features_dir=WindowsPath('C:/Users/alexy/Documents/ChatGPT_projects/LIDAR RORAIMA/artifacts/features'), models_dir=WindowsPath('C:/Users/alexy/Documents/ChatGPT_projects/LIDAR RORAIMA/artifacts/models'), inference_dir=WindowsPath('C:/Users/alexy/Documents/ChatGPT_projects/LIDAR RORAIMA/artifacts/inference'), random_seed=42)

In [2]:
import pandas as pd
from lidar_roraima.features import load_feature_tables
from lidar_roraima.inference import run_inference
from lidar_roraima.ensemble import majority_vote_classification

features = load_feature_tables(cfg.features_dir)
model_paths = [
    cfg.models_dir / "landcover_baseline.joblib",
    cfg.models_dir / "landcover_random_forest.joblib",
    cfg.models_dir / "landcover_boosting.joblib",
]
preds = [run_inference(path, features, prediction_column="pred_landcover", uncertainty_column="uncertainty_landcover_single") for path in model_paths if path.exists()]
ensemble = majority_vote_classification(preds, pred_col="pred_landcover")
ensemble.to_parquet(cfg.inference_dir / "landcover_ensemble.parquet", index=False)
ensemble.head()


,tile_id,grid_x,grid_y,pred_landcover,agreement_landcover
0,NP_T-0005,56012,33505,4,0.666667
1,NP_T-0005,56012,33506,4,0.666667
2,NP_T-0005,56012,33507,4,0.666667
3,NP_T-0005,56011,33508,4,0.666667
4,NP_T-0005,56011,33509,4,0.666667
